# 01 — Matrix Product Unitaries: basics and conventions

This notebook introduces the objects both libraries work with.

An open-boundary **Matrix Product Unitary** (MPU) on $N$ sites of local dimension $d$ is

$$U_N=\sum_{v_0,\dots,v_N=0}^{\chi-1}\ell_{v_0}\,r_{v_N}\;A[v_0,v_1]\otimes A[v_1,v_2]\otimes\cdots\otimes A[v_{N-1},v_N],$$

where each $A[v,u]$ is a $d\times d$ matrix, $\chi$ is the bond dimension and $\ell, r\in\mathbb C^\chi$ are boundary vectors.

**Storage convention (both libraries).** A site tensor is a NumPy array `A[p_in, p_out, bond_L, bond_R]` of shape `(d, d, chi, chi)`.
The block `A[:, :, v, u]` is the *transpose* of the operator $A[v,u]$ (whose rows are outputs). Boundary vectors are contracted without complex conjugation.

In [1]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")  # TreeMPU's optimizers run on the CPU

import numpy as np

from TreeMPU import UniformMPU, check_mpo_unitarity, dense_unitary_from_tensors

np.set_printoptions(precision=4, suppress=True)

## A first example: the MCZ MPU

The multi-controlled-Z example (thesis, `sec:beyond-qca`) has $d=\chi=2$ and is diagonal in the physical basis.
For physical state $|1\rangle$ the bond matrix is $\mathbb 1$; for $|0\rangle$ it is the projector $|0\rangle\langle0|$. With $\ell=(1,1)$ and $r=(1,-2)$:
* on $|1\cdots1\rangle$ the bond product is $\mathbb 1$ and the amplitude is $\ell^\top r=-1$;
* any $0$ projects the bond onto $|0\rangle$, giving $\ell_0r_0=1$.

Hence $U_N=\mathbb 1-2\,|1\cdots1\rangle\langle1\cdots1|$, the multi-controlled $Z$ on $N$ qubits.

In [2]:
A = np.zeros((2, 2, 2, 2), dtype=complex)  # A[p_in, p_out, bond_L, bond_R]
A[0, 0] = np.diag([1, 0])  # physical |0>: bond projector |0><0|
A[1, 1] = np.eye(2)  # physical |1>: bond identity
l_vec = np.array([1, 1], dtype=complex)
r_vec = np.array([1, -2], dtype=complex)

N = 3
U = dense_unitary_from_tensors([A] * N, l_vec, r_vec)  # U[out, in], site 0 least significant
print(np.real(np.diag(U)))
print("unitary:", np.allclose(U.conj().T @ U, np.eye(2**N)))

[ 1.  1.  1.  1.  1.  1.  1. -1.]
unitary: True


`dense_unitary_from_tensors` builds the $d^N\times d^N$ matrix. The qubit order is little-endian with site 0 as the least significant digit, the same as the circuits Qiskit produces. Use it only for small $N$, since memory grows as $d^{2N}$.

## Certifying unitarity for every $N$

Checking finitely many $N$ does not prove unitarity in general. The thesis gives an explicit counterexample that is unitary at $N=1,2$ but not at $N=3$. For a translation-invariant MPU, however, the **finite-size cutoff theorem** (thesis, `sec:finite-size-cutoff`) states that exact unitarity for all $N\le\chi^2+1$ implies it for every $N$.

`check_mpo_unitarity(N_max, A, l, r)` checks, for $N=1,\dots,N_{\max}$,
* spectral flatness $\mathcal P_N=(\operatorname{Tr}\rho)^2/(d^N\operatorname{Tr}\rho^2)=1$ (unitary up to scale), and
* normalization $\operatorname{Tr}\rho=d^N$, with $\rho=U_NU_N^\dagger$.

It raises `ValueError` otherwise.

In [3]:
chi = A.shape[2]
check_mpo_unitarity(chi**2 + 1, A, l_vec, r_vec)  # passes silently: certified for every N
print(f"certified for all N (checked up to chi^2+1 = {chi**2 + 1})")

# A scaled identity is unitary only up to a factor: the normalization catches it.
A_bad = np.eye(2).reshape(2, 2, 1, 1) / np.sqrt(2)
try:
    check_mpo_unitarity(3, A_bad, np.ones(1), np.ones(1))
except ValueError as err:
    print("rejected:", err)

certified for all N (checked up to chi^2+1 = 5)
rejected: Unitarity lost at N=1: U U^dag is proportional to the identity but Tr(U U^dag)/d^N = 0.5 != 1.


`UniformMPU(..., light=True)` runs exactly this certificate, up to $\chi^2+1$ sites whatever $N$ is (cheap for small $\chi$, expensive for large $\chi$), and records it:

In [4]:
mpu = UniformMPU(A=A, l_vec=l_vec, r_vec=r_vec, N=64, light=True)
print(
    "checked up to N =",
    mpu.unitarity_checked_up_to,
    "| certified for all N:",
    mpu.unitarity_certified_all_N,
)

checked up to N = 5 | certified for all N: True


## A second example: the CNOT staircase witness

`mpu_generators` provides the translation-invariant group automaton $A[g,h]=|h\rangle\langle g^{-1}h|$. For $G=\mathbb Z_2$ and $\ell=e_0$ it is the CNOT-staircase MPU. The tensor-first sampling paper uses it as an exact witness of chiral causal behaviour: support is exactly local in one direction and unbounded in the other.

In [5]:
from mpu_generators import UniformMPUGenerator

A_cnot, l_cnot, r_cnot = UniformMPUGenerator(seed=0).group_automaton(group="Zn", n=2)
U2 = dense_unitary_from_tensors([A_cnot] * 2, l_cnot, r_cnot)
print(np.real(U2))  # a CNOT (permutation matrix of order 2)

[[1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]]


**Next:** notebook 02 shows how `mpu_generators` samples *random* MPUs that are exactly unitary by construction.